# Food Image to Recipe Generator Demo

This notebook demonstrates how to use the trained model to generate recipes from food images.

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Add parent directory to path to import project modules
sys.path.append(os.path.dirname(os.getcwd()))

# Import project modules
from data.preprocessor import RecipeDataPreprocessor
from model.model import FoodImageToRecipeModel

In [ ]:
# Check for CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Preprocessor and Model

In [ ]:
# Configuration
output_dir = '../output'  # Update with your output directory
checkpoint_path = os.path.join(output_dir, 'checkpoints', 'best_model.pth')  # Or use final_model.pth
vocab_path = os.path.join(output_dir, 'vocabulary.pkl')

# Image size should match what was used during training
image_size = 224

In [ ]:
# Initialize preprocessor and load vocabulary
preprocessor = RecipeDataPreprocessor(image_size=image_size)
if os.path.exists(vocab_path):
    preprocessor.load_vocab(vocab_path)
    print(f"Vocabulary loaded with size {preprocessor.vocab_size}")
else:
    print(f"Vocabulary file not found at {vocab_path}")

In [ ]:
# Load model
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Get model parameters
    embed_size = checkpoint.get('embed_size', 512)
    hidden_size = checkpoint.get('hidden_size', 512)
    num_layers = checkpoint.get('num_layers', 1)
    
    # Initialize model
    model = FoodImageToRecipeModel(
        embed_size=embed_size,
        hidden_size=hidden_size,
        vocab_size=preprocessor.vocab_size,
        num_layers=num_layers
    ).to(device)
    
    # Load model weights
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
        
    print(f"Model loaded from {checkpoint_path}")
    model.eval()  # Set to evaluation mode
else:
    print(f"Model checkpoint not found at {checkpoint_path}")

## Generate Recipe from Image

In [ ]:
def generate_recipe_from_image(image_path):
    """Generate a recipe from a food image"""
    # Display the image
    img = Image.open(image_path).convert('RGB')
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Input Food Image')
    plt.show()
    
    # Preprocess image
    image = preprocessor.preprocess_image(image_path)
    image = image.unsqueeze(0).to(device)  # Add batch dimension
    
    # Generate recipe
    with torch.no_grad():
        sampled_ids, _ = model(image)
        
    # Convert indices to recipe text
    recipe = preprocessor.decode_recipe(sampled_ids[0].cpu())
    
    print("\nGenerated Recipe:")
    print("-" * 60)
    print(recipe)
    print("-" * 60)
    
    return recipe

In [ ]:
# Test with an example image
# Replace with your image path
image_path = '../data/test_images/pizza.jpg'

if os.path.exists(image_path):
    recipe = generate_recipe_from_image(image_path)
else:
    print(f"Image not found at {image_path}")

## Try with Your Own Images

In [ ]:
# You can upload your own image
from IPython.display import display, FileUpload

def process_uploaded_image(change):
    # Save the uploaded file
    filename = list(change['new'].keys())[0]
    content = change['new'][filename]['content']
    
    # Save the image to a temporary file
    temp_path = 'temp_upload.jpg'
    with open(temp_path, 'wb') as f:
        f.write(content)
    
    # Generate and display recipe
    generate_recipe_from_image(temp_path)

# Create upload widget
uploader = FileUpload(accept='image/*', multiple=False)
uploader.observe(process_uploaded_image, names='value')
display(uploader)

## Batch Generation

In [ ]:
def generate_recipes_for_directory(image_dir, output_dir=None):
    """Generate recipes for all images in a directory"""
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    # Get all image files
    image_extensions = ('.jpg', '.jpeg', '.png')
    image_files = [f for f in os.listdir(image_dir) 
                   if f.lower().endswith(image_extensions)]
    
    print(f"Found {len(image_files)} images in {image_dir}")
    
    for i, img_file in enumerate(image_files):
        img_path = os.path.join(image_dir, img_file)
        print(f"\nProcessing image {i+1}/{len(image_files)}: {img_file}")
        
        # Generate recipe
        recipe = generate_recipe_from_image(img_path)
        
        # Save recipe to file if output_dir provided
        if output_dir:
            recipe_file = os.path.splitext(img_file)[0] + '.txt'
            recipe_path = os.path.join(output_dir, recipe_file)
            
            with open(recipe_path, 'w') as f:
                f.write(recipe)
            print(f"Recipe saved to {recipe_path}")

In [ ]:
# Example usage for batch processing
# Uncomment and modify paths as needed

# image_dir = '../data/test_images'
# output_dir = '../data/generated_recipes'
# generate_recipes_for_directory(image_dir, output_dir)

## Analyzing Model Performance

In [ ]:
# Load training history if available
import json
import glob

# Find the latest history file
history_files = glob.glob(os.path.join(output_dir, 'logs', '*_history.json'))
if history_files:
    latest_history = max(history_files, key=os.path.getctime)
    print(f"Loading training history from {latest_history}")
    
    with open(latest_history, 'r') as f:
        history = json.load(f)
    
    # Plot training and validation loss
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['epochs'], history['train_loss'], label='Train Loss')
    if 'val_loss' in history and history['val_loss']:
        plt.plot(history['epochs'], history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    
    # Plot learning rate if available
    if 'learning_rate' in history and history['learning_rate']:
        plt.subplot(1, 2, 2)
        plt.plot(history['epochs'], history['learning_rate'])
        plt.xlabel('Epoch')
        plt.ylabel('Learning Rate')
        plt.title('Learning Rate Schedule')
        plt.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("No training history found")